# 🔒 SIREN PII 层级探针 Notebook（论文方法论对齐版）

本 Notebook 按 ACL 2026 论文 **《LLM Safety From Within: Detecting Harmful Content with Internal Representations》**（arXiv:2604.18519）附录 A.1 / Table 6 的方法论实现层级探针，并**直接调用仓库 `siren/` 库**（单一真源），不再内联简化逻辑。

### 与论文一致的关键点
- **表征 = mean-pooling**（论文式 2），不再用 max-pool；
- **train / val / test 三分**：C 网格 `{100,200,500,1000}` 与层权重 α_l 只在**验证集**上选（绝不碰测试集）；
- **报 Macro-F1**（论文口径），层级曲线画的是**测试集**泛化 F1；
- **判别词跨集留出**：见下。

> **❓ 为什么之前 33/36 层都是 1.000？**
> 有两个 token 级捷径，必须都消除：
> 1. **数字捷径**——正样本有数字、负样本没有 → 已修（每对正负样本共用完全相同的数字）。
> 2. **词汇捷径**——判别词（如 `SSN`）在训练/测试间共用，测试集对模型不构成分布外，第 1 层就线性可分 → **本版修复**。
>
> **本版做法（数字对齐 + 判别词跨集留出）**：
> 训练/验证判别词：PII = `SSN / Tax ID / ID Card`，Benign = `Order ID / Product SKU / Item Batch`；
> 测试判别词（训练中从未出现）：PII = `Passport No. / Driver License No.`，Benign = `Invoice No. / Tracking No.`。
> 只有词汇也跨集留出，探针才被迫读隐层真正的「这个标识符是不是 PII」抽象，而不是背下某个词。
>
> **验证手段**：附带一条**打乱标签对照探针**，修复生效时它应落在 Macro-F1 ≈ 0.5。

## 步骤 1：环境检查、克隆仓库与安装依赖

In [ ]:
!nvidia-smi

# 克隆仓库，使本 Notebook 可 `import siren`（单一真源；不再内联算法逻辑）
import os
if not os.path.isdir('siren-pii-probing'):
    !git clone -q https://github.com/jackyluo-learning/siren-pii-probing.git
%cd siren-pii-probing

!pip install --quiet torch transformers scikit-learn matplotlib datasets tqdm scipy

## 步骤 2：构建数字对齐 + 判别词跨集留出的 PII Benchmark（train/val/test）

In [ ]:
import numpy as np
from siren.pii_dataset_generator import build_layerwise_pii_benchmark

# 三分数据集：train/val 共用一组判别词与模板；test 用一组**训练中从未出现**的判别词与模板。
# 每个正/负样本对共用同一个数字标识符，消除数字捷径。
# 规模放大到论文量级（train 1000 / val 300 / test 600）。注意：抽取阶段约需对 1900 条做前向，
# 在 L4 上约 8–12 分钟；想快速试跑可临时改回 (400, 100, 200)。
bench = build_layerwise_pii_benchmark(n_train=1000, n_val=300, n_test=600, seed=42)

train_prompts, y_train = bench.train_prompts, bench.y_train
val_prompts,   y_val   = bench.val_prompts,   bench.y_val
test_prompts,  y_test  = bench.test_prompts,  bench.y_test

print(f'Train={len(train_prompts)}  Val={len(val_prompts)}  Test={len(test_prompts)}')
print('训练判别词:', bench.meta['train_cues'])
print('测试判别词:', bench.meta['test_cues'])
print('控制项    :', bench.meta['controls'])
print('\n示例:')
print('  train PII   :', train_prompts[0])
print('  train Benign:', train_prompts[1], '  (与上一条共用数字)')
print('  test  PII   :', test_prompts[0])
print('  test  Benign:', test_prompts[1])

## 步骤 3：用 `siren.InternalStateExtractor` 提取 36 层 **Mean-Pooled** 表征（论文式 2）

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from siren import InternalStateExtractor

MODEL_NAME = 'Qwen/Qwen3-4B'  # 论文 Figure 7 原图基座（36 层，hidden=2560）
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Loading Model "{MODEL_NAME}" on {DEVICE}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32,
    device_map='auto' if DEVICE == 'cuda' else None,
    trust_remote_code=True,
)

# 用库的抽取器：自动定位所有 decoder 层，按 attention mask 做 mean-pooling（论文式 2）。
extractor = InternalStateExtractor(model=model, device=DEVICE)
num_layers = len(extractor.target_layers)

def extract_layer_features(prompts):
    feats = {l: [] for l in range(1, num_layers + 1)}
    for text in prompts:
        enc = tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=128)
        pooled = extractor.extract_sequence_pooled(enc['input_ids'], enc['attention_mask'])
        for l in range(1, num_layers + 1):
            feats[l].append(pooled[l].squeeze(0).cpu().numpy())
    return {l: np.array(v) for l, v in feats.items()}

print('Extracting mean-pooled features for Train / Val / Test...')
train_features = extract_layer_features(train_prompts)
val_features   = extract_layer_features(val_prompts)
test_features  = extract_layer_features(test_prompts)
extractor.remove_hooks()
print(f'Extracted mean-pooled activations across {num_layers} Transformer layers!')

## 步骤 4：L1 探针（C 网格在**验证集**上选）+ SIREN 跨层融合 + 打乱标签对照

- 逐层 `SafetyNeuronProbe`：C ∈ {100,200,500,1000}，按**验证集 Macro-F1** 选优（绝不碰测试集）；η=0.8 选安全神经元。
- `AdaptiveNeuronAggregator`：层权重 α_l 由**验证集**性能计算（论文口径）。
- `SirenMLPHead` + `SirenTrainer`：在 z 上训练分类头，在**测试集**上报 Macro-F1。
- **对照探针**：把训练标签打乱后重跑逐层探针，测试 Macro-F1 应回落到 ≈0.5——用于确认「不是维度过拟合刷出来的高分」。

In [ ]:
import torch
from sklearn.metrics import f1_score
from siren import SafetyNeuronProbe, AdaptiveNeuronAggregator, SirenMLPHead, SirenTrainer
from siren.probe import PAPER_C_GRID

layers = list(range(1, num_layers + 1))

def per_layer_test_f1(probe):
    """每层用选中的探针在测试集上算 Macro-F1（这是要画的泛化曲线）。"""
    return {l: float(f1_score(y_test, probe.layer_probes[l].predict(test_features[l]),
                              average='macro', zero_division=0)) for l in layers}

# 1) 逐层 L1 探针：C 网格在验证集上选，报 Macro-F1
print('Fitting layer-wise L1 probes (C grid selected on VALIDATION, Macro-F1)...')
probe = SafetyNeuronProbe(eta=0.8, c_grid=PAPER_C_GRID, average='macro')
safety_neurons, val_f1 = probe.fit_all_layers(train_features, y_train, val_features, y_val)
test_f1 = per_layer_test_f1(probe)
for l in layers:
    print(f'  Layer {l:2d}: val={val_f1[l]:.4f}  test={test_f1[l]:.4f}  '
          f'C={probe.layer_best_c[l]:g}  neurons={len(safety_neurons[l])}')

# 2) SIREN 跨层融合：α_l 用验证集性能；在 z 上训练 MLP 头
aggregator = AdaptiveNeuronAggregator(safety_neurons, val_f1)   # 权重来自验证集
z_train = aggregator.transform(train_features)
z_val   = aggregator.transform(val_features)
z_test  = aggregator.transform(test_features)

mlp = SirenMLPHead(input_dim=z_train.size(1), hidden_dim=256)
trainer = SirenTrainer(model=mlp, lr=1e-3, device=DEVICE)
trainer.fit(z_train, torch.tensor(y_train), z_val, torch.tensor(y_val), epochs=20, batch_size=16)
with torch.no_grad():
    te_probs = mlp.predict_proba(z_test.to(trainer.device))[:, 1].cpu().numpy()
siren_pii_f1 = float(f1_score(y_test, (te_probs >= 0.5).astype(int), average='macro', zero_division=0))

# 3) 打乱标签对照探针（应回落到 ~0.5）
rng = np.random.RandomState(0)
y_train_shuf = y_train.copy(); rng.shuffle(y_train_shuf)
probe_ctrl = SafetyNeuronProbe(eta=0.8, c_grid=PAPER_C_GRID, average='macro')
probe_ctrl.fit_all_layers(train_features, y_train_shuf, val_features, y_val)
ctrl_test_f1 = {l: float(f1_score(y_test, probe_ctrl.layer_probes[l].predict(test_features[l]),
                                  average='macro', zero_division=0)) for l in layers}

print('\n==================================================')
print('SIREN PII (paper-aligned) results — Macro-F1:')
print(f'  Max single-layer TEST F1 : {max(test_f1.values()):.4f}')
print(f'  SIREN aggregated TEST F1 : {siren_pii_f1:.4f}')
print(f'  z dimension (总安全神经元): {z_train.size(1)}')
print(f'  Shuffled-label control   : mean TEST F1 = {np.mean(list(ctrl_test_f1.values())):.4f} (期望 ~0.5)')
print('==================================================')

## 步骤 5：绘制层级性能曲线（测试集 Macro-F1，含打乱标签对照）

In [ ]:
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial', 'Helvetica']
plt.rcParams['axes.edgecolor'] = '#333333'
plt.rcParams['axes.linewidth'] = 1.2

x = np.arange(num_layers)
y_test_curve = np.array([test_f1[l] for l in layers])
y_ctrl_curve = np.array([ctrl_test_f1[l] for l in layers])

fig, ax = plt.subplots(figsize=(8, 5.5), dpi=300)
ax.grid(True, linestyle='-', linewidth=0.6, alpha=0.35, color='#CCCCCC', zorder=1)

# 层级探针（测试集泛化 Macro-F1）
ax.plot(x, y_test_curve, color='#C88BA8', linewidth=1.2, alpha=0.7, zorder=2)
ax.scatter(x, y_test_curve, s=38, facecolors='#D8A3BE', edgecolors='#5C2344', linewidth=1.0, alpha=0.85, zorder=3)
if num_layers > 4:
    ax.plot(x, gaussian_filter1d(y_test_curve, sigma=1.8), color='#8C2D62',
            linewidth=3.2, label='Layer-wise Probes (test)', zorder=4)

# SIREN 跨层融合（测试集 Macro-F1）
ax.axhline(y=siren_pii_f1, color='#1F77B4', linestyle='--', linewidth=2.5,
           label=f'SIREN PII (test, {siren_pii_f1:.3f})', zorder=5)

# 打乱标签对照（应贴近 0.5）
ax.plot(x, y_ctrl_curve, color='#8A98A6', linewidth=1.6, linestyle=':',
        marker='x', markersize=5, label='Shuffled-label control', zorder=4)
ax.axhline(y=0.5, color='#B0B0B0', linewidth=1.0, alpha=0.7, zorder=1)

ax.set_xlim(-1, num_layers)
ax.set_ylim(-0.05, 1.05)
ax.set_xlabel('Layer Index', fontsize=18, labelpad=8)
ax.set_ylabel('Performance (Macro-F1)', fontsize=18, labelpad=8)
ax.tick_params(axis='both', which='major', labelsize=14)

legend = ax.legend(loc='lower center', bbox_to_anchor=(0.5, 0.02), fontsize=12,
                   frameon=True, facecolor='white', edgecolor='#CCCCCC', framealpha=0.95)
legend.get_frame().set_boxstyle('round,pad=0.4')

plt.tight_layout()
plt.savefig('pii_layerwise_performance.png', dpi=300, bbox_inches='tight')
plt.show()